# Optimal Architecture all_data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, r2_score

import keras

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping,ReduceLROnPlateau


In [ ]:
# GPU Availability Check
print(f"TensorFlow version: {tf.__version__}")
gpus = tf.config.experimental.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")

# List all physical devices
physical_devices = tf.config.list_physical_devices()
print("\n📱 All Physical Devices:")
for device in physical_devices:
    print(f"  {device}")

# Check if TensorFlow is built with CUDA support
print(f"\n🔧 CUDA Support: {tf.test.is_built_with_cuda()}")

# Prevent TensorFlow from allocating all GPU memory at once
if gpus:
    try:
        # Enable memory growth for each GPU
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(f"❌ Error setting memory growth: {e}")

In [ ]:
def set_reproducible():
    np.random.seed(12345)
    random.seed(12345)
    tf.random.set_seed(12345)
    
set_reproducible()

In [ ]:
dps1200 = pd.read_csv("../../data/dps1200_all.csv")

In [ ]:
features = dps1200.iloc[:, 4:].values
labels = dps1200.iloc[:, 0].values

In [ ]:
def convertToDecade(y:int) -> int: 
    return int(str(y)[:3])

def calculate_sample_weights(y_train):

    decades = [convertToDecade(year) for year in y_train]

    unique_decades, counts = np.unique(decades, return_counts=True)
    total_samples = len(y_train)
    
    weights = {}
    for decade, count in zip(unique_decades, counts):
        weights[decade] = 1 - count/total_samples

    sample_weights = []
    for year in y_train:
        sample_weights.append(weights[convertToDecade(year)])
        
    return np.array(sample_weights)           


In [ ]:
def LeNet_model(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, input_dim):
    
    model = keras.Sequential()
    model.add(keras.layers.Input((input_dim, 1)))
    model.add(keras.layers.GaussianNoise(0.0001))

    model.add(keras.layers.Conv1D(C1_K, (C1_S), padding='valid', activation='relu'))
    model.add(keras.layers.MaxPooling1D(pool_size=2))

    model.add(keras.layers.Conv1D(C2_K, (C2_S), padding='valid', activation='relu'))
    model.add(keras.layers.MaxPooling1D(pool_size=2))

    model.add(keras.layers.Flatten())
    model.add(keras.layers.Dropout(DropoutR))

    model.add(keras.layers.Dense(DenseN, activation='relu'))
    model.add(keras.layers.Dense(1, activation='relu'))
    

    model.compile(loss=tf.keras.losses.Huber(), optimizer=keras.optimizers.Adam(learning_rate=0.004), metrics=['mean_absolute_error'])
    
    return model

In [ ]:
import datetime
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, Activation, Add,
    MaxPooling1D, GlobalAveragePooling1D, Dense, Concatenate, Dropout
)

def create_optimizer():
    # opt = keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-5)  # causes underfitting?
    # opt = keras.optimizers.Adam(learning_rate=0.001)  # worked well for sub_data
    opt = keras.optimizers.Adam(learning_rate=0.0001)  # best so far for all_data
    return opt

# best so far at loss around 3e-4
# praise be Gemini Pro 2.5, https://g.co/gemini/share/e3f26b1e78f3
def inception_residual_module(input_tensor, num_filters):
    """
    The core building block of the network. It processes data at multiple scales
    and includes a residual connection for stable training.
    """
    # Use a 1x1 convolution as a "bottleneck" to reduce computational cost
    bottleneck = Conv1D(filters=num_filters // 4, kernel_size=1, padding='same', use_bias=False)(input_tensor)
    bottleneck = BatchNormalization()(bottleneck)
    bottleneck = Activation('relu')(bottleneck)

    # --- Parallel branches for multi-scale feature extraction ---
    # Small kernel for sharp peaks
    # branch1 = Conv1D(filters=num_filters // 4, kernel_size=3, padding='same', use_bias=False)(bottleneck)
    branch1 = Conv1D(filters=num_filters // 4, kernel_size=15, padding='same', use_bias=False)(bottleneck)

    # Medium kernel for wider features
    # branch2 = Conv1D(filters=num_filters // 2, kernel_size=7, padding='same', use_bias=False)(bottleneck)
    branch2 = Conv1D(filters=num_filters // 2, kernel_size=63, padding='same', use_bias=False)(bottleneck)
    
    # Large kernel for broad shapes
    # branch3 = Conv1D(filters=num_filters // 4, kernel_size=11, padding='same', use_bias=False)(bottleneck)
    branch3 = Conv1D(filters=num_filters // 4, kernel_size=255, padding='same', use_bias=False)(bottleneck)

    # Combine the features from all branches
    concatenated = Concatenate()([branch1, branch2, branch3])
    
    # Final 1x1 convolution to restore the desired number of filters
    final_conv = Conv1D(filters=num_filters, kernel_size=1, padding='same', use_bias=False)(concatenated)
    final_conv = BatchNormalization()(final_conv)

    # --- Residual Connection ---
    # Add the original input to the output of the convoluted path.
    # This helps prevent vanishing gradients in a deep network.
    if input_tensor.shape[-1] != num_filters:
        shortcut = Conv1D(filters=num_filters, kernel_size=1, padding='same', use_bias=False)(input_tensor)
        shortcut = BatchNormalization()(shortcut)
    else:
        shortcut = input_tensor
        
    output = Add()([final_conv, shortcut])
    output = Activation('relu')(output)
    
    return output

def build_wood_age_incepresnet(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, C3_K, C3_S, input_dim):
    """
    Constructs the full Inception-Residual model.
    """

    inputs = Input(shape=(input_dim, 1))

    # --- Stem: Initial Feature Extraction ---
    # A standard convolution to process the raw input and reduce dimensionality.
    
    # larger kernel for larger input size
    # sub_data -> all_data
    # input size 400 -> 1866
    # kernel size 13 -> 21 -> 31
    # x = Conv1D(filters=64, kernel_size=13, strides=2, padding='same', use_bias=False)(inputs)
    x = Conv1D(filters=64, kernel_size=63, strides=2, padding='same', use_bias=False)(inputs) 
    
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(pool_size=3, strides=2, padding='same')(x)

    # --- Core Blocks: Deep Feature Learning ---
    # Stack the custom modules to learn increasingly complex features.
    x = inception_residual_module(x, num_filters=128)
    x = inception_residual_module(x, num_filters=128)
    x = MaxPooling1D(pool_size=3, strides=2, padding='same')(x)

    x = inception_residual_module(x, num_filters=256)
    x = inception_residual_module(x, num_filters=256)
    x = MaxPooling1D(pool_size=3, strides=2, padding='same')(x)

    # --- Head: Final Prediction ---
    # Global Average Pooling is the key to fighting overfitting.
    # It creates one feature per filter, summarizing the entire spectrum.
    x = GlobalAveragePooling1D()(x)
    
    # reduce gap between train and val
    x = Dropout(DropoutR)(x)
    
    # A small dense head for the final regression output.
    x = Dense(64, activation='relu')(x)
    outputs = Dense(1, activation='linear', name='age_output')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss=tf.keras.losses.Huber(), optimizer=create_optimizer(), metrics=['mean_absolute_error'])
    # model.summary()

    return model

# Create TensorBoard callback
def create_tensorboard_callback(time_string, log_name, i):
    return tf.keras.callbacks.TensorBoard(log_dir=f'tb_logs/{log_name}/{time_string}/it_{i}')

# Early stopping to prevent overfitting
def create_early_stopping():
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=200,
        restore_best_weights=True,
        verbose=1
    )
    return early_stopping

# Add learning rate scheduling callback
def create_lr_scheduler():
    # reduces LR when validation loss plateaus
    scheduler = tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=80,
        min_lr=1e-6,
        verbose=1
    )
    return scheduler

# Crossvalidation

Since NN training involves random sampling and weights initialization (in this case), it is useful to use cross-validation.

In [ ]:
def recalculate_year(years):
    # return np.exp(years)
    return years

In [ ]:
## Compute error metrics
def error_metrices(y_true_train, y_predicted_train, y_true_test, y_predicted_test):
    y_true_train = recalculate_year(y_true_train)
    y_predicted_train = recalculate_year(y_predicted_train)
    y_true_test = recalculate_year(y_true_test)
    y_predicted_test = recalculate_year(y_predicted_test)
    
    rmse_train = np.sqrt(mean_squared_error(y_true_train, y_predicted_train))
    rmse_test = np.sqrt(mean_squared_error(y_true_test, y_predicted_test))
    R2_train= r2_score(y_true_train, y_predicted_train)
    R2_test= r2_score(y_true_test, y_predicted_test)
    h = tf.keras.losses.Huber()
    hub_train = h(y_true_train, y_predicted_train).numpy()
    hub_test = h(y_true_test, y_predicted_test).numpy()

    print('*********** Benchmark results ***********\n')
    print(f"R2    (Train/Test) = {R2_train:.3f} / {R2_test:.3f}")
    print(f"RMSE  (Train/Test) = {rmse_train:.3f} / {rmse_test:.3f}")
    print(f"Huber (Train/Test) = {hub_train:.3f} / {hub_test:.3f}")

    return (rmse_train, rmse_test, R2_train, R2_test, hub_train, hub_test)

class ModelWithData:
    def __init__(self, history, train_x, train_label, train_predicted, test_x, test_label, test_predicted, score:float, iteration, model) -> None:
        self.history = history
        self.train_x = train_x
        self.train_label = train_label
        self.train_predicted = train_predicted
        self.test_x = test_x
        self.test_label = test_label
        self.test_predicted = test_predicted
        self.score = score
        self.iteration = iteration
        self.model = model

    def isBetter(self, otherScore: float) -> bool:
        return otherScore < 0 or (self.score >= 0 and self.score <= otherScore)

In [ ]:

def evaluations_of_models(features, labels, DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S):
    import math
    batch_size = 45
    
    x = np.array(features, dtype=np.float32)  # Convert to float32 upfront
    y_int = np.array(labels)
    y = y_int.astype(np.float32)  # Use float32
    # y = np.log(y)  # Apply log transformation for better numerical stability
    input_dim = 1866
    epochs = 1000

    # Define the number of folds for Cross-Validation
    n_splits = 5
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    scores_train = { "rmse": [], "r2": [], "huber": []}
    scores_test = { "rmse": [], "r2": [], "huber": []}

    # monitor = EarlyStopping(monitor='val_loss', min_delta=4e-5, patience=50, verbose=0, mode='auto', restore_best_weights=True)
    # rdlr = ReduceLROnPlateau(patience=25, factor=0.5, min_lr=1e-6, monitor='val_loss', verbose=1)

    bestMwd = ModelWithData(None, 0, 0, 0, 0, 0, 0, -1, 0, None)
    
    x_full_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_full_tensor = tf.convert_to_tensor(y, dtype=tf.float32)
    
    time_string = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        
    # Iterate through the models
    i = 0
    for train_index, test_index in skf.split(x, y_int):
        i = i + 1
        print(f'\n\n> Iteration {i}')

        # generate model
        # model = LeNet_model(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, input_dim)
        # log_name = 'lenet_all'
        model = build_wood_age_incepresnet(DenseN, DropoutR, C1_K, C1_S, C2_K, C2_S, C3_K=63, C3_S=1, input_dim=input_dim)
        log_name = 'incepresnet_all_diluted'
        # model.summary()

        callbacks = [create_tensorboard_callback(time_string, log_name, i)]
        callbacks += [create_lr_scheduler()]
        callbacks += [create_early_stopping()]

        x_train, x_test = x[train_index], x[test_index]
        y_train, y_test = y[train_index], y[test_index]
        
        x_train_tensor = tf.gather(x_full_tensor, train_index)
        y_train_tensor = tf.gather(y_full_tensor, train_index)
        x_test_tensor = tf.gather(x_full_tensor, test_index)
        y_test_tensor = tf.gather(y_full_tensor, test_index)
        
        sample_weight = calculate_sample_weights(y_int[train_index])
        sample_weight_tensor = tf.convert_to_tensor(sample_weight, dtype=tf.float32)
        weight_dataset = tf.data.Dataset.from_tensor_slices(sample_weight_tensor)
        
        train_dataset_with_weights = tf.data.Dataset.zip((
            tf.data.Dataset.from_tensor_slices(x_train_tensor),
            tf.data.Dataset.from_tensor_slices(y_train_tensor), 
            weight_dataset
        ))
        
        train_dataset_with_weights = train_dataset_with_weights.shuffle(buffer_size=len(train_index))
        train_dataset_with_weights = train_dataset_with_weights.batch(batch_size)
        train_dataset_with_weights = train_dataset_with_weights.prefetch(tf.data.AUTOTUNE)

        val_dataset = tf.data.Dataset.from_tensor_slices((x_test_tensor, y_test_tensor))
        val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        
       # Training the CNN-Model on X_train_fold, y_train_fold
        history = model.fit(train_dataset_with_weights, epochs=epochs, batch_size=45, 
                            validation_data=val_dataset, verbose=0, callbacks=callbacks)

        # Evaluation of the model on X_test_fold, y_test_fold
        evaluation_results = model.evaluate(val_dataset, verbose=0)
        evaluation_labels = model.metrics_names

        for j in range(len(evaluation_results)):
            print(f'{evaluation_labels[j]}: {evaluation_results[j]}')

        # Analyse error metrics
        train_pred_dataset = tf.data.Dataset.from_tensor_slices(x_train_tensor).batch(batch_size)
        test_pred_dataset = tf.data.Dataset.from_tensor_slices(x_test_tensor).batch(batch_size)
        train_pred = model.predict(train_pred_dataset, verbose=0)
        test_pred = model.predict(test_pred_dataset, verbose=0)

        (rmse_train, rmse_test, r2_train, r2_test, huber_train, huber_test) = error_metrices(y_train, train_pred, y_test, test_pred)

        mwd = ModelWithData(history, x_train, y_train, train_pred, x_test, y_test, test_pred, rmse_test, i, model)

        if mwd.isBetter(bestMwd.score):
            bestMwd = mwd

        scores_train["rmse"].append(rmse_train)
        scores_train["r2"].append(r2_train)
        scores_train["huber"].append(huber_train)

        scores_test["rmse"].append(rmse_test)
        scores_test["r2"].append(r2_test)
        scores_test["huber"].append(huber_test)

        ## clear session 
        tf.keras.backend.clear_session()

    scores_train_mean = {}
    scores_test_mean = {}
    num_model = len(scores_train["rmse"])

    for metric in scores_train.keys():
        scores_train_mean[metric] = np.mean(scores_train[metric])
        scores_test_mean[metric] = np.mean(scores_test[metric])
        print(f'Train: {metric} (mean of {num_model} models)= {scores_train_mean[metric]} \nTest: {metric} (mean of {num_model} models)= {scores_test_mean[metric]}')

    return bestMwd

In [ ]:
# bestMwd = evaluations_of_models(features, labels, DenseN=422, DropoutR=0.3494832761988253, C1_K=24, C1_S=34, C2_K=52, C2_S=97)
bestMwd = evaluations_of_models(features, labels, DenseN=422, DropoutR=0.0, 
# bestMwd = evaluations_of_models(features, labels, DenseN=422, DropoutR=0.1, 
                                # C1_K=55, C1_S=45, C2_K=43, C2_S=36)
                                C1_K=55, C1_S=45, C2_K=43, C2_S=36)
print(f"Best model is from iteration {bestMwd.iteration}")

In [ ]:
plt.plot(bestMwd.history.history['loss'], label='Training Loss')
plt.plot(bestMwd.history.history['val_loss'], label='Validation Loss')

plt.yscale('log')
plt.ylabel('Huber Loss')
plt.xlabel('Epochs')
plt.legend()

Eval_of_model = plt.legend()

In [ ]:
plt.plot(bestMwd.history.history['loss'], label='Training Loss')
plt.plot(bestMwd.history.history['val_loss'], label='Validation Loss')

plt.yscale('log')
plt.ylabel('Huber Loss')
plt.xlabel('Epochs')
plt.legend()

Eval_of_model_2 = plt.legend()

In [ ]:
x_values = np.linspace(min(labels), max(labels), 100)

# Plot the line of equality (y=x)
plt.plot(x_values, x_values, color='blue', linestyle='--', label='Y = X line')

# undo log transformation for better interpretability
x_train_label = recalculate_year(bestMwd.train_label)
x_test_label = recalculate_year(bestMwd.test_label)
y_train_predicted = recalculate_year(bestMwd.train_predicted)
y_test_predicted = recalculate_year(bestMwd.test_predicted)

# Scatter plot for predicted values
plt.scatter(x_train_label, y_train_predicted, c='k', label='Train Labels')
plt.scatter(x_test_label, y_test_predicted, c='r', label='Test Labels')

# Labels, legend, and title
plt.xlabel('True Labels')
plt.ylabel('Predicted Labels')
plt.legend()
plt.title('Scatter Plot of True vs Predicted Labels with Line of Equality')

# Show the plot
plt.show()

# Save the model

In [ ]:
bestMwd.model.save('dps1200all_model.keras')

# Check if working

In [ ]:
dps1200all_model = tf.keras.models.load_model('dps1200all_model.keras')

# Show the model architecture
dps1200all_model.summary()

# Evaluation of the restored model

In [ ]:
evaluation_results = dps1200all_model.evaluate(features, labels, verbose=0)
evaluation_labels = dps1200all_model.metrics_names

for j in range(len(evaluation_results)):
    print(f'{evaluation_labels[j]}: {evaluation_results[j]}')


In [ ]:
def error_metrices_restored_model(y_true, y_predicted):
    rmse_train = np.sqrt(mean_squared_error(y_true, y_predicted))
    R2_train= r2_score(y_true, y_predicted)
    h = tf.keras.losses.Huber()
    hub_train = h(y_true, y_predicted).numpy()
    
    print('*********** Benchmark results ***********\n')
    print(f"R2    = {R2_train:.3f}")
    print(f"RMSE  = {rmse_train:.3f}")
    print(f"Huber = {hub_train:.3f}")

In [ ]:
restored_pred = dps1200all_model.predict(features, verbose=0)

# undo log transformation for better interpretability
restored_pred = recalculate_year(restored_pred)

In [ ]:
error_metrices_restored_model(labels, restored_pred)